# Project 7 — Part A (v2 High-Accuracy): Vision Agent Ensemble Training (Colab / Kaggle GPU)

**Owner:** Person A (Vision Specialist)  
**Architecture:** 4-Fold Stratified **EfficientNet-V2-S** (`IMAGENET1K_V1`, `288×288` aspect-preserving) + Label Smoothing + Mixup + Clean Cooldown + Horizontal-Flip **Test-Time Augmentation (TTA)** + Ensemble Grad-CAM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdo0412/decentralized-insurance-fraud-detection/blob/main/notebooks/01_vision_cnn_training.ipynb)

### Instructions:
1. In Google Colab, go to **Runtime → Change runtime type → T4 GPU**.
2. Click **Runtime → Run all** (~6–8 minutes total with FP16 Mixed Precision).
3. Cell 7 packages and downloads `part_a_deliverables_v2.zip` to your computer.

In [ ]:
# 1. Clone (or update) the repository & verify GPU
import os
import torch

REPO_URL = "https://github.com/abdo0412/decentralized-insurance-fraud-detection.git"
REPO_DIR = "decentralized-insurance-fraud-detection"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git fetch origin && git reset --hard origin/main

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name    :", torch.cuda.get_device_name(0))

In [ ]:
# 2. Download the Kaggle Car Damage Severity Dataset (No token prompt needed!)
import os
import shutil
import kagglehub

TARGET_DIR = "data/raw/vehicle_damage"
os.makedirs(TARGET_DIR, exist_ok=True)

if not os.path.exists(os.path.join(TARGET_DIR, "data3a")):
    print("Downloading dataset via kagglehub (no token required)...")
    downloaded_path = kagglehub.dataset_download("prajwalbhamere/car-damage-severity-dataset")
    print("Downloaded to:", downloaded_path)
    for item in os.listdir(downloaded_path):
        src = os.path.join(downloaded_path, item)
        dst = os.path.join(TARGET_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print("Dataset ready in", TARGET_DIR)
else:
    print("Dataset already present in", TARGET_DIR)

In [ ]:
# 3. Verify the Deduplicated, Leak-Free v2 Split Manifests
import pandas as pd

held_out_df = pd.read_csv("data/processed/held_out_images.csv")
cv_df = pd.read_csv("data/processed/cnn_splits/cv_folds.csv")
test_df = pd.read_csv("data/processed/cnn_splits/test.csv")

print(f"Held-Out Pool (for Person C, untouched by CNN): {len(held_out_df)} unique images")
print(f"CNN 4-Fold CV Pool: {len(cv_df)} unique images | Untouched Test Set: {len(test_df)} unique images")

cnn_paths = set(cv_df["image_path"]).union(test_df["image_path"])
held_out_paths = set(held_out_df["image_path"])
assert len(cnn_paths.intersection(held_out_paths)) == 0, "DATA LEAKAGE DETECTED!"
assert len(set(cv_df["image_path"]).intersection(set(test_df["image_path"]))) == 0, "TEST LEAKAGE DETECTED!"
print("Zero overlap & zero duplicate leakage verified!")

In [ ]:
# 4. Train 4-Fold EfficientNet-V2-S Ensemble + Evaluate on Untouched Test Split
!python scripts/train_vision_cnn.py \
    --splits-dir data/processed/cnn_splits \
    --backbone efficientnet_v2_s \
    --image-size 288 \
    --num-folds 4 \
    --epochs-warmup 4 \
    --epochs-finetune 18 \
    --cooldown-epochs 4 \
    --batch-size 32 \
    --lr-warmup 1e-3 \
    --lr-finetune 1.5e-4 \
    --label-smoothing 0.08 \
    --mixup-alpha 0.2

In [ ]:
# 5. Display Upgraded Test Metrics, Confusion Matrix, and Ensemble Grad-CAM Grid
!git pull origin main
!python scripts/train_vision_cnn.py --splits-dir data/processed/cnn_splits --backbone efficientnet_v2_s --image-size 288 --eval-only

import json
from IPython.display import Image as IPyImage, display

with open("report/figures/vision_metrics.json", "r") as f:
    metrics = json.load(f)

print(f"Model Version        : {metrics.get('model_version', metrics.get('backbone'))}")
print(f"Mean CV Val Macro F1 : {metrics.get('cv_val_macro_f1_mean', 0):.4f} (folds: {metrics.get('fold_val_macro_f1s', [])})")
print(f"Test Accuracy (TTA)  : {metrics['test_accuracy']:.4f}")
print(f"Test Macro F1 (TTA)  : {metrics['test_macro_f1']:.4f}")

display(IPyImage(filename="report/figures/vision_confusion_matrix.png"))
display(IPyImage(filename="report/figures/vision_gradcam_examples.png"))

In [ ]:
# 6. Score the Held-Out Photos for Person C's Decision Agent
!python scripts/predict_held_out.py \
    --claims-csv data/processed/held_out_images.csv \
    --weights models/vision_efficientnet_v2_s.pt \
    --output-csv data/processed/vision_predictions.csv

preds_df = pd.read_csv("data/processed/vision_predictions.csv")
preds_df.head(10)

In [ ]:
# 7. Package & Download Upgraded v2 Deliverables
!zip -r part_a_deliverables_v2.zip \
    models/vision_efficientnet_v2_s.pt \
    report/figures/vision_confusion_matrix.png \
    report/figures/vision_gradcam_examples.png \
    report/figures/vision_metrics.json \
    data/processed/vision_predictions.csv

try:
    from google.colab import files
    files.download("part_a_deliverables_v2.zip")
except ImportError:
    print("Saved part_a_deliverables_v2.zip in the working directory!")